In [1]:
# n = 10
[1,-1,-1,1]
# predicted return, return volatility over prior year, correlation of stock with CRSP value-weighted index (Mkt-RF), OOS R2
# long only strategy cum return - up to 2020
# sharpe ratio is (monthly excess return)/volatility of all dates -> actual; volatility is monthly std over all years
# use logit to get probs for all stocks, select top n highest probs to form 1/n portfolio 

[1, -1, -1, 1]

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize, approx_fprime
import torch
import matplotlib.pyplot as plt

from scipy.stats import mstats

In [2]:
link_table = pd.read_csv('permno-comnam.csv', index_col=0)
link_table.set_index('permno', inplace=True)

In [3]:
green_vars = pd.read_csv("green cleaned.csv", dtype={'ncusip': 'string'})
green_vars['ret_fwd_1'] = (green_vars.groupby('permno')['ret_excess'].shift(-1) )
green_vars = green_vars.dropna(subset=["ret_fwd_1"])

green_vars['datadate'] = pd.to_datetime(green_vars['DATE'], format='%Y%m%d')+pd.tseries.offsets.MonthEnd(0)
green_vars = green_vars.set_index('permno', drop=False)
green_vars = green_vars[['datadate', 'mve_m', 'ret_fwd_1']]

In [4]:
print(green_vars)

         datadate         mve_m  ret_fwd_1
permno                                    
10006  1980-01-31  3.034208e+05  -0.067695
10006  1980-02-29  3.676485e+05  -0.184178
10006  1980-03-31  3.410715e+05  -0.040051
10006  1980-04-30  2.823806e+05   0.076577
10006  1980-05-31  2.746290e+05  -0.013676
...           ...           ...        ...
93436  2024-07-31  6.321554e+08  -0.082191
93436  2024-08-31  7.413801e+08   0.217942
93436  2024-09-30  6.840044e+08  -0.048925
93436  2024-10-31  8.390474e+08   0.377469
93436  2024-11-30  8.020335e+08   0.166308

[267678 rows x 3 columns]


In [5]:
green_vars[green_vars['datadate']=='2023-11-30'].sort_values(by='mve_m', ascending=False).head(10).loc[93436]

datadate     2023-11-30 00:00:00
mve_m                638454482.0
ret_fwd_1               0.030688
Name: 93436, dtype: object

## Value weighted

In [6]:
portfolio_returns = []
portfolio_turnover = []
prev_weights_dict = {}
prev_returns_dict = {}
prev_gross_return = 0.0
transaction_cost = 0.001  # 50 bps, same as your nodewise backtest

print_firms = True

# --- ADDED: Initialize a list to hold the weight records ---
weights_records = []
# -----------------------------------------------------------

# 3.5 is -39:-8
for date in sorted(green_vars['datadate'].unique())[-39:-8]:
    # Get data for this date
    date_data = green_vars[green_vars['datadate'] == date].copy()
    
    # Remove rows with missing returns or missing market values
    date_data = date_data.dropna(subset=['ret_fwd_1', 'mve_m'])
    
    if len(date_data) > 0 and date_data['mve_m'].sum() > 0:
        # Calculate weights (market value / total market value)
        weights = date_data['mve_m'] / date_data['mve_m'].sum()
        
        # Create weights dictionary
        weights_dict = weights.to_dict()
        returns_dict = date_data['ret_fwd_1'].to_dict()

        # --- ADDED: Append this period's weights to our records ---
        dt_str = date.strftime('%Y-%m-%d') if hasattr(date, 'strftime') else str(date)
        for permno, weight in weights_dict.items():
            weights_records.append({
                'date': dt_str,
                'permno': permno,
                'weight': weight
            })
        # ----------------------------------------------------------
        
        if print_firms:
            w = weights.to_frame(name='weight')
            merged = w.merge(link_table, on='permno', how='left').sort_values(by='weight', ascending=False).head(10)
            print(f"\n{date}")
            print(merged)
        
        # Calculate gross value-weighted return
        gross_return = (weights * date_data['ret_fwd_1']).sum()
        
        # Calculate transaction costs
        if len(prev_weights_dict) > 0:
            # Get all assets (current + previous)
            all_assets = set(weights_dict.keys()) | set(prev_weights_dict.keys())
            
            # Adjust previous weights: w+_{t,j} = w_{t,j} * (1 + r_{t,j}) / (1 + r_p,t)
            adjusted_prev = {}
            for asset in all_assets:
                prev_w = prev_weights_dict.get(asset, 0.0)
                
                if asset in prev_returns_dict and prev_gross_return > -0.99:
                    prev_r = prev_returns_dict[asset]
                    adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
                else:
                    adjusted_prev[asset] = 0.0
            
            # Renormalize adjusted weights (only over current assets)
            adj_sum = sum(adjusted_prev.get(a, 0.0) for a in weights_dict.keys())
            if adj_sum > 1e-10:
                adjusted_prev_normalized = {k: adjusted_prev.get(k, 0.0)/adj_sum 
                                           for k in weights_dict.keys()}
            else:
                adjusted_prev_normalized = {k: 0.0 for k in weights_dict.keys()}
            
            # Turnover: sum over all assets
            turnover = sum(abs(weights_dict.get(a, 0.0) - adjusted_prev_normalized.get(a, 0.0)) 
                          for a in all_assets)
            
            # Transaction cost
            tc = transaction_cost * (1 + gross_return) * turnover
        else:
            # First period: full turnover
            turnover = sum(abs(w) for w in weights_dict.values())
            tc = transaction_cost * turnover
        
        # Net return
        net_return = gross_return - tc
        
        portfolio_returns.append(net_return)
        portfolio_turnover.append(turnover)
        
        # Update for next iteration
        prev_weights_dict = weights_dict.copy()
        prev_returns_dict = returns_dict.copy()
        prev_gross_return = gross_return

# Convert to array
portfolio_returns = np.array(portfolio_returns)
portfolio_turnover = np.array(portfolio_turnover)

# Calculate Sharpe ratio
mean_return = portfolio_returns.mean()
std_return = portfolio_returns.std()
sharpe_ratio = mean_return / std_return if std_return > 0 else 0
sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)

print(f"\n{'='*60}")
print("S&P 500 PERFORMANCE (WITH TRANSACTION COSTS)")
print(f"{'='*60}")
print(f"Mean Monthly Return:      {mean_return:.4f}")
print(f"Monthly Std Dev:          {std_return:.4f}")
print(f"Monthly Sharpe Ratio:     {sharpe_ratio:.4f}")
print(f"Annualized Sharpe Ratio:  {sharpe_ratio_annualized:.4f}")
print(f"Average Turnover:         {portfolio_turnover.mean():.4f}")
print(f"Number of periods:        {len(portfolio_returns)}")
print(f"{'='*60}")


2021-10-31 00:00:00
          weight               comnam
permno                               
14593   0.068183            APPLE INC
10107   0.062089       MICROSOFT CORP
84788   0.048856       AMAZON COM INC
13407   0.023655   META PLATFORMS INC
93436   0.022839            TESLA INC
86580   0.015192          NVIDIA CORP
47896   0.014348  JPMORGAN CHASE & CO
22111   0.012471    JOHNSON & JOHNSON
55976   0.011401          WALMART INC
92611   0.011027             VISA INC

2021-11-30 00:00:00
          weight                  comnam
permno                                  
10107   0.068546          MICROSOFT CORP
14593   0.067661               APPLE INC
84788   0.047086          AMAZON COM INC
93436   0.030800               TESLA INC
13407   0.021079      META PLATFORMS INC
86580   0.017611             NVIDIA CORP
47896   0.013976     JPMORGAN CHASE & CO
92655   0.011953  UNITEDHEALTH GROUP INC
22111   0.011805       JOHNSON & JOHNSON
55976   0.011471             WALMART INC

2021-12-3


2023-03-31 00:00:00
          weight                  comnam
permno                                  
14593   0.075028               APPLE INC
10107   0.059726          MICROSOFT CORP
84788   0.031082          AMAZON COM INC
93436   0.020938               TESLA INC
86580   0.018447             NVIDIA CORP
11850   0.014561        EXXON MOBIL CORP
92655   0.014282  UNITEDHEALTH GROUP INC
47896   0.013573     JPMORGAN CHASE & CO
22111   0.012840       JOHNSON & JOHNSON
13407   0.012526      META PLATFORMS INC

2023-04-30 00:00:00
          weight                  comnam
permno                                  
14593   0.080784               APPLE INC
10107   0.066804          MICROSOFT CORP
84788   0.033013          AMAZON COM INC
86580   0.021377             NVIDIA CORP
93436   0.020486               TESLA INC
13407   0.014631      META PLATFORMS INC
11850   0.013909        EXXON MOBIL CORP
92655   0.013736  UNITEDHEALTH GROUP INC
22111   0.012577       JOHNSON & JOHNSON
55976   0.01238

In [7]:
def save_weights_from_df(df, filename):
    records = []
    # Iterate through the dates and weight dictionaries in the dataframe
    for _, row in df.iterrows():
        dt_str = row['date'].strftime('%Y-%m-%d')
        w_dict = row['portfolio_weights']
        
        # Unpack the dictionary into individual rows
        for permno, weight in w_dict.items():
            records.append({
                'date': dt_str,
                'permno': permno,
                'weight': weight
            })
    
    # Save to CSV
    if records:
        weights_df = pd.DataFrame(records)
        weights_df.to_csv(filename, index=False)
        print(f"Saved {len(weights_df)} weight records to {filename}")
    else:
        print(f"No weights to save for {filename}")

print("\nExporting portfolio weights...")
# --- ADDED: Save the recorded weights to CSV at the very end ---
print("\nExporting portfolio weights...")
weights_df = pd.DataFrame(weights_records)
filename = 'sp500_portfolio_weights.csv'
weights_df.to_csv(filename, index=False)
print(f"Saved {len(weights_df)} weight records to {filename}")
# ---------------------------------------------------------------


Exporting portfolio weights...

Exporting portfolio weights...
Saved 13832 weight records to sp500_portfolio_weights.csv


In [10]:
# Save value-weighted portfolio returns locally
pd.DataFrame({
    'portfolio_returns_vw': portfolio_returns
}).to_csv('portfolio_returns_vw 10 years 4o.csv', index=False)

print('Saved portfolio_returns to portfolio_returns 10 years 4o.csv')

Saved portfolio_returns to portfolio_returns 10 years 4o.csv


## Equal weighted

In [22]:
portfolio_returns_ew = []
portfolio_turnover_ew = []
prev_weights_dict = {}
prev_returns_dict = {}
prev_gross_return = 0.0
transaction_cost = 0.001  # 10 bps per unit of turnover; match the value-weighted setup if you prefer
print_firms = True

for date in sorted(green_vars['datadate'].unique())[-8-31:-8]:
    # Get data for this date
    date_data = green_vars[green_vars['datadate'] == date].copy()
    
    # Remove rows with missing returns or missing market values
    date_data = date_data.dropna(subset=['ret_fwd_1', 'mve_m'])
    
    if len(date_data) > 0:
        # Calculate equal weights (1/N for each firm)
        n_firms = len(date_data)
        weights = pd.Series(1 / n_firms, index=date_data.index)
        weights_dict = weights.to_dict()
        returns_dict = date_data['ret_fwd_1'].to_dict()
        
        if print_firms:
            w = weights.rename('weight').reset_index()
            merged = w.merge(link_table, on='permno', how='left').sort_values(by='weight', ascending=False).head(10)
            print(date)
            print(merged)
        
        # Equal-weighted gross return
        gross_return = date_data['ret_fwd_1'].mean()
        
        # Transaction costs based on turnover from prior period to current equal-weight target
        if len(prev_weights_dict) > 0:
            all_assets = set(weights_dict.keys()) | set(prev_weights_dict.keys())
            
            adjusted_prev = {}
            for asset in all_assets:
                prev_w = prev_weights_dict.get(asset, 0.0)
                
                if asset in prev_returns_dict and prev_gross_return > -0.99:
                    prev_r = prev_returns_dict[asset]
                    adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
                else:
                    adjusted_prev[asset] = 0.0
            
            adj_sum = sum(adjusted_prev.get(a, 0.0) for a in weights_dict.keys())
            if adj_sum > 1e-10:
                adjusted_prev_normalized = {
                    k: adjusted_prev.get(k, 0.0) / adj_sum
                    for k in weights_dict.keys()
                }
            else:
                adjusted_prev_normalized = {k: 0.0 for k in weights_dict.keys()}
            
            turnover = sum(
                abs(weights_dict.get(a, 0.0) - adjusted_prev_normalized.get(a, 0.0))
                for a in all_assets
            )
            tc = transaction_cost * (1 + gross_return) * turnover
        else:
            # First period: full turnover into the initial portfolio
            turnover = sum(abs(w) for w in weights_dict.values())
            tc = transaction_cost * turnover
        
        net_return = gross_return - tc
        portfolio_returns_ew.append(net_return)
        portfolio_turnover_ew.append(turnover)
        
        # Update for next iteration
        prev_weights_dict = weights_dict.copy()
        prev_returns_dict = returns_dict.copy()
        prev_gross_return = gross_return

# Convert to array
portfolio_returns_ew = np.array(portfolio_returns_ew)
portfolio_turnover_ew = np.array(portfolio_turnover_ew)

# Calculate Sharpe ratio (assuming monthly returns)
mean_return = portfolio_returns_ew.mean()
std_return = portfolio_returns_ew.std()
sharpe_ratio = mean_return / std_return if std_return > 0 else 0

# Annualized Sharpe ratio (multiply by sqrt(12) for monthly data)
sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)

print(f"Mean Monthly Return: {mean_return:.4f}")
print(f"Monthly Std Dev: {std_return:.4f}")
print(f"Monthly Sharpe Ratio: {sharpe_ratio:.4f}")
print(f"Annualized Sharpe Ratio: {sharpe_ratio_annualized:.4f}")
print(f"Average Turnover: {portfolio_turnover_ew.mean():.4f}")
print(f"Number of periods: {len(portfolio_returns_ew)}")

2021-10-31 00:00:00
     permno    weight                  comnam
408   90071  0.002232        N R G ENERGY INC
409   90199  0.002232       C B R E GROUP INC
410   90215  0.002232          SALESFORCE INC
411   90248  0.002232       DOMINOS PIZZA INC
412   90386  0.002232           MOSAIC CO NEW
413   90454  0.002232   MARKETAXESS HLDGS INC
414   90455  0.002232  MONOLITHIC PWR SYS INC
415   90505  0.002232    LAS VEGAS SANDS CORP
400   89393  0.002232             NETFLIX INC
401   89508  0.002232              CARMAX INC
2021-11-30 00:00:00
     permno    weight                       comnam
443   93436  0.002252                    TESLA INC
0     10104  0.002252                  ORACLE CORP
1     10107  0.002252               MICROSOFT CORP
2     10138  0.002252       T ROWE PRICE GROUP INC
3     10145  0.002252  HONEYWELL INTERNATIONAL INC
4     10516  0.002252    ARCHER DANIELS MIDLAND CO
5     10696  0.002252                   FISERV INC
6     10909  0.002252                  CERNER 

In [19]:
# Save equal-weighted portfolio returns locally
pd.DataFrame({
    'portfolio_returns_ew': portfolio_returns_ew
}).to_csv('portfolio_returns_ew 10 years 3.5.csv', index=False)

print('Saved portfolio_returns_ew to portfolio_returns_ew 10 years 3.5.csv')

Saved portfolio_returns_ew to portfolio_returns_ew 10 years 3.5.csv


# Daily Data

In [3]:
df = pd.read_csv('./Paper revisions/filtered_daily_data_w_ticker.csv')
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)
df = df.rename(columns={'date': 'datadate'})

In [4]:
df

,permno,datadate,ret_excess,ticker,ret_fwd_1
0,10104,2023-04-21,0.003194,ORCL,0.002868
1,10104,2023-04-24,0.002868,ORCL,-0.014639
2,10104,2023-04-25,-0.014639,ORCL,-0.003901
3,10104,2023-04-26,-0.003901,ORCL,0.014013
4,10104,2023-04-27,0.014013,ORCL,-0.003547
...,...,...,...,...,...
166391,93436,2023-06-30,0.016393,TSLA,0.068734
166392,93436,2023-07-03,0.068734,TSLA,-0.029156
166393,93436,2024-04-10,-0.029156,TSLA,0.016325
166394,93436,2024-04-11,0.016325,TSLA,-0.020542


In [8]:
# 1. Ensure 'datadate' is a datetime object in both dataframes
df['datadate'] = pd.to_datetime(df['datadate'])
green_vars['datadate'] = pd.to_datetime(green_vars['datadate'])

# 2. Reset index on green_vars if 'permno' is currently set as the index
if 'permno' not in green_vars.columns:
    green_vars = green_vars.reset_index()

# 3. Create the 'year_month' merge key
# For daily data, extract the current month
df['year_month'] = df['datadate'].dt.to_period('M')

# For monthly data, shift the month FORWARD by 1. 
# This ensures that end-of-January mve_m (Month 1) becomes Month 2, 
# perfectly aligning the previous month's market cap with the current month's daily returns.
green_vars['year_month'] = green_vars['datadate'].dt.to_period('M') + 1

# 4. Merge the dataframes
# Subsetting green_vars avoids bringing in duplicate columns (like ret_fwd_1)
merged_df = pd.merge(
    df,
    green_vars[['permno', 'year_month', 'mve_m']],
    on=['permno', 'year_month'],
    how='left'
)

# Clean up the helper column
merged_df = merged_df.drop(columns=['year_month'])

In [14]:
merged_df = merged_df.set_index('permno')
merged_df

,datadate,ret_excess,ticker,ret_fwd_1,mve_m
permno,,,,,
10104,2023-04-21,0.003194,ORCL,0.002868,2.356525e+08
10104,2023-04-24,0.002868,ORCL,-0.014639,2.356525e+08
10104,2023-04-25,-0.014639,ORCL,-0.003901,2.356525e+08
10104,2023-04-26,-0.003901,ORCL,0.014013,2.356525e+08
10104,2023-04-27,0.014013,ORCL,-0.003547,2.356525e+08
...,...,...,...,...,...
93436,2023-06-30,0.016393,TSLA,0.068734,5.207812e+08
93436,2023-07-03,0.068734,TSLA,-0.029156,6.463569e+08
93436,2024-04-10,-0.029156,TSLA,0.016325,6.429454e+08


In [19]:
portfolio_returns = []
portfolio_turnover = []
prev_weights_dict = {}
prev_returns_dict = {}
prev_gross_return = 0.0
transaction_cost = 0.001  # 50 bps, same as your nodewise backtest

print_firms = True

# 3.5 is -39:-8
for date in sorted(merged_df['datadate'].unique())[-62:]:
    # Get data for this date
    date_data = merged_df[merged_df['datadate'] == date].copy()
    
    # Remove rows with missing returns or missing market values
    date_data = date_data.dropna(subset=['ret_fwd_1', 'mve_m'])
    
    if len(date_data) > 0 and date_data['mve_m'].sum() > 0:
        # Calculate weights (market value / total market value)
        weights = date_data['mve_m'] / date_data['mve_m'].sum()
        
        # Create weights dictionary
        weights_dict = weights.to_dict()
        returns_dict = date_data['ret_fwd_1'].to_dict()
        
        if print_firms:
            w = weights.to_frame(name='weight')
            merged = w.merge(link_table, on='permno', how='left').sort_values(by='weight', ascending=False).head(10)
            print(f"\n{date}")
            print(merged)
        
        # Calculate gross value-weighted return
        gross_return = (weights * date_data['ret_fwd_1']).sum()
        
        # Calculate transaction costs
        if len(prev_weights_dict) > 0:
            # Get all assets (current + previous)
            all_assets = set(weights_dict.keys()) | set(prev_weights_dict.keys())
            
            # Adjust previous weights: w+_{t,j} = w_{t,j} * (1 + r_{t,j}) / (1 + r_p,t)
            adjusted_prev = {}
            for asset in all_assets:
                prev_w = prev_weights_dict.get(asset, 0.0)
                
                if asset in prev_returns_dict and prev_gross_return > -0.99:
                    prev_r = prev_returns_dict[asset]
                    adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
                else:
                    adjusted_prev[asset] = 0.0
            
            # Renormalize adjusted weights (only over current assets)
            adj_sum = sum(adjusted_prev.get(a, 0.0) for a in weights_dict.keys())
            if adj_sum > 1e-10:
                adjusted_prev_normalized = {k: adjusted_prev.get(k, 0.0)/adj_sum 
                                           for k in weights_dict.keys()}
            else:
                adjusted_prev_normalized = {k: 0.0 for k in weights_dict.keys()}
            
            # Turnover: sum over all assets
            turnover = sum(abs(weights_dict.get(a, 0.0) - adjusted_prev_normalized.get(a, 0.0)) 
                          for a in all_assets)
            
            # Transaction cost
            tc = transaction_cost * (1 + gross_return) * turnover
        else:
            # First period: full turnover
            turnover = sum(abs(w) for w in weights_dict.values())
            tc = transaction_cost * turnover
        
        # Net return
        net_return = gross_return - tc
        
        portfolio_returns.append(net_return)
        portfolio_turnover.append(turnover)
        
        # Update for next iteration
        prev_weights_dict = weights_dict.copy()
        prev_returns_dict = returns_dict.copy()
        prev_gross_return = gross_return

# Convert to array
portfolio_returns = np.array(portfolio_returns)
portfolio_turnover = np.array(portfolio_turnover)

# Calculate Sharpe ratio
mean_return = portfolio_returns.mean()
std_return = portfolio_returns.std()
sharpe_ratio = mean_return / std_return if std_return > 0 else 0
sharpe_ratio_annualized = sharpe_ratio * np.sqrt(252)

print(f"\n{'='*60}")
print("S&P 500 PERFORMANCE (WITH TRANSACTION COSTS)")
print(f"{'='*60}")
print(f"Mean Daily Return:      {mean_return:.4f}")
print(f"Daily Std Dev:          {std_return:.4f}")
print(f"Daily Sharpe Ratio:     {sharpe_ratio:.4f}")
print(f"Annualized Sharpe Ratio:  {sharpe_ratio_annualized:.4f}")
print(f"Average Turnover:         {portfolio_turnover.mean():.4f}")
print(f"Number of periods:        {len(portfolio_returns)}")
print(f"{'='*60}")


2024-02-01 00:00:00
          weight                  comnam
permno                                  
14593   0.080316               APPLE INC
10107   0.075412          MICROSOFT CORP
84788   0.041990          AMAZON COM INC
86580   0.033005             NVIDIA CORP
93436   0.021354               TESLA INC
13407   0.021117      META PLATFORMS INC
50876   0.014932          LILLY ELI & CO
47896   0.013269     JPMORGAN CHASE & CO
92655   0.013139  UNITEDHEALTH GROUP INC
55976   0.011452             WALMART INC

2024-02-02 00:00:00
          weight                  comnam
permno                                  
14593   0.081029               APPLE INC
10107   0.076082          MICROSOFT CORP
84788   0.042363          AMAZON COM INC
86580   0.033298             NVIDIA CORP
93436   0.021544               TESLA INC
13407   0.021304      META PLATFORMS INC
50876   0.015064          LILLY ELI & CO
47896   0.013387     JPMORGAN CHASE & CO
92655   0.013256  UNITEDHEALTH GROUP INC
55976   0.01155